# 01 — Data Loading & Cleaning Checks

Exploratory notebook for the **chomkar-decision-grid** synthetic dataset (DERA-ZN stage **D — Detect** territory).

> **Ground rule:** notebooks are *experiments only*. The deterministic tools in `tools/` are the source of truth — this notebook **imports** them rather than re-implementing any check. See [docs/data_dictionary.md](../docs/data_dictionary.md) and [AGENTS.md](../AGENTS.md).

What we look at here:
1. Load the five CSVs and eyeball row/column counts
2. Schema check against the columns the tools require
3. Blank / missing-value scan
4. Run the real Detect audit (`validate_data.build_detect`) on every order
5. Supply vs demand — the **volume gap** picture
6. A quick farmer-population profile

In [1]:
import os, sys
from collections import defaultdict

# Import the repo's deterministic tools (source of truth)
TOOLS = os.path.abspath(os.path.join(os.getcwd(), "..", "tools"))
if TOOLS not in sys.path:
    sys.path.insert(0, TOOLS)

import config
import dataio
import validate_data as vd


def show_table(headers, rows):
    """Tiny stdlib table printer (no pandas needed)."""
    rows = [tuple(str(c) for c in r) for r in rows]
    widths = [max([len(str(h))] + [len(r[i]) for r in rows]) for i, h in enumerate(headers)]
    line = " | ".join(str(h).ljust(w) for h, w in zip(headers, widths))
    print(line)
    print("-" * len(line))
    for r in rows:
        print(" | ".join(c.ljust(w) for c, w in zip(r, widths)))

print("tools imported from:", TOOLS)

tools imported from: C:\Users\Admin\chomkar-decision-grid\tools


## 1. Load the dataset

In [2]:
farmers = dataio.load_farmers()
orders = dataio.load_orders()
transport = dataio.load_transport()
weather = dataio.load_weather()
prices = dataio.load_prices()

for name, rows in [("farmers", farmers), ("buyer_orders", orders), ("transport_costs", transport),
                   ("weather_sample", weather), ("market_prices", prices)]:
    print(f"{name:>16}: {len(rows):>3} rows x {len(rows[0])} columns")

         farmers:  26 rows x 14 columns
    buyer_orders:   5 rows x 11 columns
 transport_costs:  40 rows x 8 columns
  weather_sample: 120 rows x 8 columns
   market_prices:  24 rows x 6 columns


## 2. Schema check

Columns required by the Detect tool (`validate_data.REQUIRED`) must all be present.

In [3]:
tables = {"farmers": farmers, "orders": orders, "transport": transport, "weather": weather}
for name, rows in tables.items():
    missing = [c for c in vd.REQUIRED[name] if c not in rows[0]]
    print(f"{name:>10}: {'OK — all required columns present' if not missing else 'MISSING ' + str(missing)}")

   farmers: OK — all required columns present
    orders: OK — all required columns present
 transport: OK — all required columns present
   weather: OK — all required columns present


## 3. Blank / missing-value scan

In [4]:
for name, rows in [("farmers", farmers), ("orders", orders), ("transport", transport),
                   ("weather", weather), ("prices", prices)]:
    blanks = defaultdict(int)
    for r in rows:
        for k, v in r.items():
            if str(v).strip() == "":
                blanks[k] += 1
    print(f"{name:>10}: {dict(blanks) if blanks else 'no blank values'}")

   farmers: no blank values
    orders: no blank values
 transport: no blank values
   weather: no blank values
    prices: no blank values


## 4. Run the real Detect audit on every order

This calls the same pure function the pipeline uses: `validate_data.build_detect` — schema, ranges, referential checks, and the **volume-gap** blocker.

In [5]:
rows = []
for o in orders:
    res = vd.build_detect(o["order_id"], farmers, orders, transport, weather)
    rows.append((o["order_id"],
                 "CLEAN" if res["clean"] else "BLOCKED",
                 ", ".join(b["code"] for b in res["blockers"]) or "—",
                 ", ".join(w["code"] for w in res["warnings"]) or "—"))
show_table(("order", "verdict", "blockers", "warnings"), rows)

order     | verdict | blockers   | warnings
-------------------------------------------
order_001 | CLEAN   | —          | —       
order_002 | CLEAN   | —          | —       
order_003 | CLEAN   | —          | —       
order_004 | BLOCKED | VOLUME_GAP | —       
order_005 | CLEAN   | —          | —       


## 5. Supply vs demand — the volume-gap picture

The core Chomkar problem: single farmers declare sub-ton amounts, buyers need 1–3 tons. Quality matters too — only farmers at/above the required grade count as *qualifying* supply.

In [6]:
supply_total = defaultdict(int)
for f in farmers:
    supply_total[f["declared_crop"]] += int(f["declared_qty_kg"])

rows = []
for o in orders:
    need = int(o["quantity_kg"])
    total = supply_total[o["commodity"]]
    qualifying = sum(int(f["declared_qty_kg"]) for f in farmers
                     if f["declared_crop"] == o["commodity"]
                     and config.grade_meets(f["quality_grade"], o["quality_required"]))
    rows.append((o["order_id"], o["commodity"], f">={o['quality_required']}", need, total, qualifying,
                 "fillable" if qualifying >= need else f"VOLUME GAP (short {need - qualifying} kg)"))
show_table(("order", "commodity", "grade", "needs_kg", "declared_kg", "qualifying_kg", "verdict"), rows)

order     | commodity     | grade | needs_kg | declared_kg | qualifying_kg | verdict                   
-------------------------------------------------------------------------------------------------------
order_001 | bok_choy      | >=B   | 1500     | 2550        | 2550          | fillable                  
order_002 | morning_glory | >=B   | 1200     | 2000        | 1650          | fillable                  
order_003 | cabbage       | >=A   | 1500     | 2500        | 1650          | fillable                  
order_004 | long_bean     | >=B   | 3000     | 1350        | 1350          | VOLUME GAP (short 1650 kg)
order_005 | cucumber      | >=B   | 1000     | 1700        | 1700          | fillable                  


## 6. Farmer-population profile

In [7]:
by_prov = defaultdict(lambda: {"n": 0, "kg": 0, "rel": []})
for f in farmers:
    p = by_prov[f["province"]]
    p["n"] += 1
    p["kg"] += int(f["declared_qty_kg"])
    p["rel"].append(float(f["reliability_score"]))

rows = [(prov, v["n"], v["kg"], f"{sum(v['rel'])/len(v['rel']):.2f}") for prov, v in sorted(by_prov.items())]
show_table(("province", "farmers", "declared_kg", "avg_reliability"), rows)

grades = defaultdict(int)
for f in farmers:
    grades[f["quality_grade"]] += 1
print("\ngrade distribution:", dict(sorted(grades.items())))
qty = sorted(int(f["declared_qty_kg"]) for f in farmers)
print(f"declared qty per farmer: min {qty[0]} kg, median {qty[len(qty)//2]} kg, max {qty[-1]} kg")
print("\n=> every farmer is sub-ton — several must combine to fill any 1-3 t order (the volume gap).")

province     | farmers | declared_kg | avg_reliability
------------------------------------------------------
Kampong Cham | 8       | 3450        | 0.82           
Kandal       | 6       | 2500        | 0.82           
Siem Reap    | 6       | 2600        | 0.73           
Takeo        | 6       | 2600        | 0.75           

grade distribution: {'A': 9, 'B': 15, 'C': 2}
declared qty per farmer: min 300 kg, median 450 kg, max 600 kg

=> every farmer is sub-ton — several must combine to fill any 1-3 t order (the volume gap).


## Findings

- All five CSVs match the required schema; no blank values in the synthetic seed.
- Detect verdicts: `order_004` is **BLOCKED** (`VOLUME_GAP` — long_bean supply 1,350 kg < 3,000 kg needed); all other orders pass Detect.
- Grade-aware supply matters: e.g. morning_glory has 2,000 kg declared but only 1,650 kg at grade ≥ B.
- Every farmer declares 200–600 kg, so consolidated lots are always multi-farmer — exactly the problem the DERA-ZN grid decides.

**Next notebook:** [02_order_feasibility.ipynb](02_order_feasibility.ipynb) — candidate-lot assembly and feasibility.